## Initialization

In [ ]:
# Imports
from typing import Callable, Literal
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    EnergyColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.helpers import (
    get_input_with_default,
    stop
)

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

## Experiment ID Input

In [ ]:
experiment_ids = [
    "TB-zero_threshold",
    "TB-twentyfive_again_threshold",
    "TB-fifty_threshold",
    "TB-seventyfive_threshold",
    "TB-hundred_threshold",
    "TB-hundred_twentyfive_threshold",
    "TB-hundred_fifty_threshold",
    "TB-hundred_sevetyfive_threshold",  # spelling is correct
    "TB-twohundred_threshold",
    "TB-twohundred_twentyfive_threshold",
    "TB-twohundred_fifty_threshold",
    "TB-twohundred_seventyfive_threshold",
    "TB-threehundred_threshold",
    "TB-fourhundred_threshold",
]

In [ ]:
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Neutron Classification

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

## Histogram Creation

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    print(exp_id)
    
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    neutrons_only = psd_report.query(n_class_col_name).copy()

    print(psd_report.shape)
    print(neutrons_only.shape)

    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    signals_df = exp_data["signals_df"].astype("int32")

    signals_np = signals_df.to_numpy()
    baselines = signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    signals_np = -signals_np + baselines
    signals_df = pd.DataFrame(signals_np, index=signals_df.index, columns=signals_df.columns)
    psd_report["peak_height"] = signals_df.max(axis=1)
    exp_data["signals_df"] = signals_df

In [ ]:
height_bin_width = 100
energy_bin_width = 50
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    energy = psd_report["ENERGY"]
    peak_height = psd_report["peak_height"]

    height_bins = np.arange(0, 15500, step=height_bin_width)
    energy_bins = np.arange(0, 3500, step=energy_bin_width)

    Zh, *_ = np.histogram(peak_height, bins=height_bins)
    Ze, *_ = np.histogram(energy, bins=energy_bins)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
        "all": {
            "height": Zh,
            "height_bins": height_bins,
            "energy": Ze,
            "energy_bins": energy_bins
        },
    }

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    phd_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["all"]
    for key in ["height", "energy"]:
        Z = phd_data[key]
        Z_total = np.sum(Z)
        Z_fraction = Z / Z_total
        phd_data[f"{key}_fraction"] = Z_fraction

## Figure Base Data

In [ ]:
dl_folder = Path.home() / "Downloads" / "neutron_detection_paper"
dl_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_figure_data() -> pd.DataFrame:
    """
    Contains histogram data for neutron detector noise
    The histogram shows the pulse height distribution
    The index is the bin midpoint, and the series values are the counts.
    """
    dataset_whitelist = [
        "TB-twentyfive_again_threshold",
        "TB-fifty_threshold",
        "TB-seventyfive_threshold",
        "TB-hundred_threshold",
        "TB-hundred_twentyfive_threshold",
        "TB-hundred_fifty_threshold",
        "TB-hundred_sevetyfive_threshold",  # spelling is correct
        "TB-twohundred_threshold",
        "TB-twohundred_fifty_threshold",
        "TB-threehundred_threshold",
        "TB-fourhundred_threshold",
    ]
    name_map = {
        "TB-twentyfive_again_threshold": 25,
        "TB-fifty_threshold": 50,
        "TB-seventyfive_threshold": 75,
        "TB-hundred_threshold": 100,
        "TB-hundred_twentyfive_threshold": 125,
        "TB-hundred_fifty_threshold": 150,
        "TB-hundred_sevetyfive_threshold": 175,
        "TB-twohundred_threshold": 200,
        "TB-twohundred_fifty_threshold": 250,
        "TB-threehundred_threshold": 300,
        "TB-fourhundred_threshold": 400,
    }

    histo_series = []
    for exp_id, exp_data in experiment_neutron_data.items():
        if exp_id not in dataset_whitelist:
            continue

        phd_histogram_data = exp_data[
            ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["all"]
        bins = phd_histogram_data["height_bins"]
        histo_counts = phd_histogram_data["height"]
        name = name_map[exp_id]
    
        energy_bin_mids = (bins[1:] + bins[:-1]) / 2

        series = pd.Series(
            data=histo_counts,
            index=energy_bin_mids,
            name=name
        )
        histo_series.append(series)
    df = pd.concat(histo_series, axis=1)
    return df

## Plotting

### Plot Style Constants

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"

### Plot Functions 

In [ ]:
def plot_figure(ax: mpl.axes.Axes):
    zorders = {
        25: 0,
        50: 0,
        75: 0,
        100: 0,
        125: 0,
        150: 0,
        175: 0,
        200: 0,
        250: 1,
        300: 0,
        400: 0,
    }
    df = get_figure_data()

    for name, series in df.items():
        color = bg_blue if name == 250 else bg_red
        alpha = 1 if name == 250 else 0.5
        zorder_mod = zorders[name]
        ax.fill_between(
            series.index,
            series,
            label=name,
            alpha=alpha,
            zorder=5+zorder_mod,
            color=color
        )
    ax.set_xlim(0, 15500)
    ax.set_ylim(0, 5000)
    ax.tick_params(labelsize=fontsize)
    ax.set_xlabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax.set_ylabel("Counts", fontsize=fontsize)

### Plot Creation

In [ ]:
fig_folder = dl_folder / "figures"
fig_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 8),
    dpi=600,
    layout="constrained"
)
plot_figure(ax)
fig.savefig(fig_folder / "si_fig_3a.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
input("Processing done, hit Enter to finish")
stop()

## Base Data Export

In [ ]:
excel_folder = dl_folder / "excel"
excel_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def export_figure_data():
    df = get_figure_data()
    df.to_excel(
        excel_folder / "si_figure_3a.xlsx",
        index_label="Pulse height (ADC channel)"
    )

In [ ]:
export_figure_data()

In [ ]:
input("Processing done, hit Enter to finish")
stop()